<a href="https://colab.research.google.com/github/Steco17/jsc-api-backend/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JSC Translator - full training run

Fine-tunes NLLB-200-distilled-600M on all 29 `data_ready` Cameroonian languages (see `data/languages.json`) using LoRA.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

**One-time setup - GitHub token (repo is private):**
1. Create a token at https://github.com/settings/tokens with `repo` scope.
2. In this notebook, click the key icon (Secrets) in the left sidebar.
3. Add a secret named `GITHUB_TOKEN` with that token as the value, and enable notebook access.

**If this session disconnects partway through training:** just reconnect (`Runtime > Reconnect`) and re-run all cells from the top (`Runtime > Run all`). Checkpoints are saved to Google Drive every `--eval-steps` steps, and the training cell auto-resumes from the latest one - it does not start over.

**On runtime:** this is a big run (1.69M rows, 29 languages). The first training cell logs its steps/sec after ~50 steps - check that early and extrapolate before assuming it'll finish in one session. It very likely won't; multiple reconnect-and-resume cycles are expected and fine.

In [20]:
!nvidia-smi

Mon Aug  3 19:51:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
from google.colab import drive
drive.mount('/content/drive')

# Checkpoints and the final merged model live in Drive, not the ephemeral
# Colab VM disk, so they survive a disconnect.
OUT_DIR = '/content/drive/MyDrive/jsc_translator/model_out'
import os
os.makedirs(OUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'Steco17/jsc-api-backend'

import os
if not os.path.isdir('/content/jsc_api_backend'):
    !git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git /content/jsc_api_backend
%cd /content/jsc_api_backend

Cloning into '/content/jsc_api_backend'...
remote: Write access to repository not granted.
fatal: unable to access 'https://github.com/Steco17/jsc-api-backend.git/': The requested URL returned error: 403
[Errno 2] No such file or directory: '/content/jsc_api_backend'
/content


In [23]:
!pip install -q -r requirements-train.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-train.txt'


In [24]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU"

torch 2.11.0+cu128 | cuda available: True


In [25]:
# Regenerate the combined dataset from the raw CSVs (not committed to git -
# it's fully derivable and would have bloated the repo).
!python scripts/prepare_all.py data -o data/prepared

python3: can't open file '/content/scripts/prepare_all.py': [Errno 2] No such file or directory


In [26]:
NEW_LANGS = (
    'agq_Latn ags_Latn azo_Latn bbj_Latn bbk_Latn bfd_Latn bmo_Latn bri_Latn '
    'bsq_Latn bum_Latn bwt_Latn etu_Latn ewo_Latn fub_Latn gya_Latn ksf_Latn '
    'lem_Latn lmp_Latn lns_Latn mcu_Latn mgo_Latn muy_Latn nge_Latn oku_Latn '
    'pny_Latn tui_Latn vut_Latn xmg_Latn yat_Latn'
)

# batch/grad-accum tuned for a T4's 16GB (local smoke testing used batch=2
# on a 4GB card). Watch the first ~50 steps' it/s in the output below, then
# extrapolate total runtime - adjust --epochs down if it won't fit your
# available session time.
!python scripts/finetune.py \
  --train data/prepared/train.jsonl \
  --dev data/prepared/dev.jsonl \
  --new-langs {NEW_LANGS} \
  --out {OUT_DIR} \
  --epochs 4 --batch 16 --grad-accum 4 --eval-steps 500

python3: can't open file '/content/scripts/finetune.py': [Errno 2] No such file or directory


## After training finishes

`<OUT_DIR>/merged/` (the `OUT_DIR` set two cells above, under `/content/drive/MyDrive/jsc_translator/model_out`) now holds the full fine-tuned model - already in Drive, already safe. Convert it to CTranslate2 int8 for CPU serving:

In [27]:
CT2_DIR = '/content/drive/MyDrive/jsc_translator/model_ct2'
!ct2-transformers-converter --model {OUT_DIR}/merged --output_dir {CT2_DIR} --quantization int8

/bin/bash: line 1: ct2-transformers-converter: command not found


Download `model_out/merged/` and `model_ct2/` from Drive to run `app/main.py` locally (`MODEL_DIR`/`TOKENIZER_DIR` env vars), or run `scripts/evaluate.py` against `data/prepared/test.jsonl` first to check quality per language pair before deploying.

Once you're happy with it, flip each trained language's status from `data_ready` to `fine_tuned` in `data/languages.json` and commit - that's what makes `app/main.py` actually expose them via `/translate`.